In [1]:
#%%     Imports

%load_ext autoreload
%autoreload 2
%matplotlib qt5

# Standard Imports
import tensorflow as tf
import keras_tuner as kt 
import kormos

tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float64')
tf_float = 'float64'
tf.random.set_seed(42)

import numpy as np
np_float = np.float64

import os
import datetime

# Own imports
import ContinuumMechanics as CM
import layers
import subANNs
import Outputs
import Plots
import Callbacks
import utils
import fit
import build


device = 'gpu:' + str(0) if tf.test.is_gpu_available() else 'cpu:0'

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


In [ ]:
dataset = 'anisotropic_multistep'

load_model = True
modelToLoad = '.\\Results_{:}\\20260107-005816'.format(dataset)

pathToData = '.\\datasets\\{:}_data\\'.format(dataset)   

if load_model: 
    outputFolder = modelToLoad
else:
    date = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    outputFolder = '.\\Results_{:}\\'.format(dataset) + date

if not os.path.exists(outputFolder):
    os.makedirs(outputFolder)


In [ ]:


incomp = True
visco = True

uncoupled = True # must be True, was only experementally tested
rateDependent = False # if relaxation times are strain rate dependent

numDir = 3 # number $n$ of preferred material directions $\vec{m}$ (e.g. fibers) to be learned by the model
numTens = 1  # number $R$ of generalized Maxwell models $\tilde{\tns{L}}_r$ to be learned by the model
numExtra = 0 # number $n_f$ of features which parameterize the free energy functions and relaxation times
numExtraStruc = 0 # number of features which parameterize the preferred material direction $\vec{m}_i$ and corresponding wieghts w_i^{(r)}

# for saving and loading    
custom_objects = {
    'dirModelOrtho': CM.dirModelOrtho,
    'weightModelOrtho': CM.weightModelOrtho,
    'PositiveHeNormal': subANNs.PositiveHeNormal,
    'PositiveVarianceScaling': subANNs.PositiveVarianceScaling,
    'PositiveGlorotNormal': subANNs.PositiveGlorotNormal,
    'PsiSigmaLayer': CM.PsiSigmaLayer, 
    'GradientLayer': CM.GradientLayer,
    'ScaleLayer': layers.ScaleLayer, 
    'stressUpdateLayer': CM.stressUpdateLayer,
    'SparsityRegularizer': utils.SparsityRegularizer
}

In [ ]:
#%% Hyperparameters
#
lr = 0.01
clipnorm = None
lambda_maxwell = 0.0 # penalty to enforce sparsity of the Prony Series

# Time scaling constants (powers of 10)
tau_min = 0
tau_max = 2

# Some activation functions
acti0 = 'tanh'
acti1 = 'sigmoid'
acti2 = 'softplus'
acti3 = 'linear'
acti4 = 'elu'
acti5 = 'squared_softplus'
acti6 = 'neg_squared_softplus'

#
EPOCHS = 10000
earlyStopPatience = 10000

# maximum number N_max of Maxwell elements per generalized Maxwell model
nMaxwell = 3

# Equilibrium free energy: number of layers / neurons per layer
layer_size_psi = [14,] # last layer of shape (1,) is automatically included in the model
activations_psi = [acti2, acti2, acti2]

# Preferred material directions: number of layers / neurons per layer
layer_size_dir = [5,]
activations_dir = [acti2, acti3, acti3]

# Weights of preferred material directions: number of layers / neurons per layer
layer_size_dir = [5,] 
activations_w = [acti2, acti3, acti2]

# Relaxation times: : number of layers / neurons per layer
layer_size_tau = [10,] # last layer of shape (1,) is automatically included in the model
activations_tau = [acti2, acti2, acti2, acti2]

# Non-equilibrium free energy: number of layers / neurons per layer
layer_size_psi_a = [10,] # last layer of shape (1,) is automatically included in the model
activations_psi_a = [acti2, acti2, acti2, acti2]

In [5]:
#%% Data
        
# Training and validation data
trainDs = tf.data.experimental.load(pathToData + "ds_train_defGrad", compression='GZIP')
valDs = tf.data.experimental.load(pathToData + "ds_train_defGrad", compression='GZIP')

tf.data.experimental.save(trainDs, outputFolder + '\\ds_train_defGrad', compression='GZIP')
tf.data.experimental.save(valDs, outputFolder + '\\ds_valid_defGrad', compression='GZIP')

trainDs_infy= tf.data.experimental.load(pathToData + "ds_train_eq_defGrad", compression='GZIP')

# instDs = tf.data.experimental.load(pathToData + "ds_inst_ref_defGrad", compression='GZIP')
# ds_step = tf.data.experimental.load(pathToData + "ds_step_defGrad", compression='GZIP')
# ds_step_ref = tf.data.experimental.load(pathToData + "ds_step_ref_defGrad", compression='GZIP')

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.
Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


In [6]:
for x,y in trainDs.take(1):
    print(x[0].shape)

(5, 1000, 3, 3)


In [27]:
#%% build the model with prescribed hyperparameters
nSteps = int(np.loadtxt(pathToData + 'n_time_steps.txt'))
# nSteps = 10
if load_model == True:
    model_fit  = Outputs.loadModel(modelToLoad, 'model_8', custom_objects)
    
else:             
    model_fit, model_full = build.build_model(nSteps,
                                                  numTens,
                                                  numDir,
                                                  nMaxwell,
                                                  numExtra,
                                                  numExtraStruc,
                                                  uncoupled,
                                                  rateDependent,
                                                  layer_size_psi,
                                                  activations_psi,
                                                  layer_size_dir,
                                                  activations_dir,
                                                  layer_size_w,
                                                  activations_w,
                                                  layer_size_tau,
                                                  activations_tau,
                                                  layer_size_psi_a, 
                                                  activations_psi_a,
                                                  [tau_min, tau_max],
                                                  lambda_maxwell,
                                                  incomp,
                                                  visco,
                                                  tf_float
                                                )
     

In [ ]:
# instantaneous elastic mode
S_infy  = model_full.get_layer('S_infy').output
F    = model_full.get_layer('F_input').input
time = model_full.get_layer('time_input').input
P_infy  = tf.keras.layers.Lambda(lambda x:  tf.matmul(x[0], x[1]))([F, S_infy])
model_full_infy = tf.keras.models.Model([F,], P_infy)

model_full_infy.trainable = True
# history, model_full_infy = fit.deterministic(model_full_infy, trainDs_infy, 100, 500, outputFolder, valDs=trainDs_infy, Maxwell_monitor=None, loss=tf.keras.losses.MeanSquaredError())
Plots.plot_elastic_stress_biaxial(model_full_infy, trainDs_infy, trainDs_infy, outputFolder)

ValueError: No such layer: S_infy. Existing layers are: ['F_input', 'dir_model_ortho_2', 'F_ref', 'C', 'L', 'weight_model_ortho_1', 'C_ref', 'C_bar', 'H', 'C_bar_ref', 'invars_I', 'invars_J', 'invars_I_ref', 'invars_J_ref', 'concat_invars', 'concat_invars_ref', 'model_psi_a_0', 'split_Psi_a_ref_0', 'dPsi_a_dI_ref_1_1', 'dPsi_a_dJ_ref_1_1', 'dPsi_a_dI_ref_1_2', 'dPsi_a_dJ_ref_1_2', 'dPsi_a_dI_ref_1_3', 'dPsi_a_dJ_ref_1_3', 'tf.__operators__.getitem_77', 'tf.__operators__.getitem_80', 'tf.__operators__.getitem_78', 'tf.__operators__.getitem_81', 'tf.__operators__.getitem_79', 'tf.__operators__.getitem_82', 'delta_a_1_1', 'delta_a_1_2', 'delta_a_1_3', 'alpha_a_1_1', 'tf.__operators__.getitem_83', 'tf.__operators__.getitem_84', 'beta_a_1_1', 'alpha_a_1_2', 'tf.__operators__.getitem_85', 'tf.__operators__.getitem_86', 'beta_a_1_2', 'alpha_a_1_3', 'tf.__operators__.getitem_87', 'tf.__operators__.getitem_88', 'beta_a_1_3', 'split_Psi_a_N_0', 'concat_Psi_a_ref_0', 'Psi_sigma_1_1', 'Psi_sigma_1

In [32]:
model_fit.load_weights(outputFolder + '\\ckpt\\ckpt-epoch-1581.ckpt')

In [124]:
model_full_infy.trainable = False

In [182]:
for x,y in trainDs_infy:
    x
w = model_full.get_layer('model_w')(x)
dir = model_full.get_layer('model_dir')(x)
L = CM.ten2_L(dir)
H = model_full.get_layer('H')([L, w, nSteps, numDir, numTens])[0,0,0] 

alpha = np.arccos( (H[0,0] -H[1,1])/(3*(H[0,0] + H[1,1])-2))/2 * 180/np.pi
w = 3/2*(H[0,0] + H[1,1]) - 1

print(alpha, w)


26.412484528954966 tf.Tensor(0.23295959676903166, shape=(), dtype=float64)


In [125]:
#%% Fit
normalized_error = False
stochastic = False

Maxwell_monitor = None # Callbacks.NonZeroWeightsMonitor(numTens, lambda_prony)
reg_callback = Callbacks.RegularizationCallback(lambda_maxwell, numTens)

if normalized_error == True:
    loss = fit.IndividuallyNormalizedLoss()  
else:
    # loss = tf.keras.losses.MeanSquaredError()
    loss = fit.WeightedLoss()


# stochastic optimizer
if stochastic:

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm, name='optimizer')
    
    #Outputs.saveOptimizerConfig(optimizer, outputFolder=outputFolder)
    #Outputs.saveModelConfig(model_fit, outputFolder=outputFolder) 
    
    # tf.keras.backend.set_value(model_fit.optimizer.learning_rate, 0.1)
    model_fit.compile(
            optimizer=optimizer,
            loss=loss,
            run_eagerly=False
        )
    
    with tf.device('/device:CPU:0'):
        history, model_fit =  fit.stochastic(model_fit, trainDs, EPOCHS, earlyStopPatience, outputFolder, valDs=valDs, Maxwell_monitor=Maxwell_monitor)
                                    
# deterministic optimizer
else:
    with tf.device('/device:CPU:0'):
        history, model_fit = fit.deterministic(model_fit, trainDs, EPOCHS, earlyStopPatience, outputFolder, valDs=valDs, Maxwell_monitor=Maxwell_monitor, loss=loss)


#%%    
Outputs.saveLoss(history, Maxwell_monitor=Maxwell_monitor, outputFolder=outputFolder)
Outputs.plotLoss(history, Maxwell_monitor=Maxwell_monitor, title='$\Lambda = {:}$'.format(lambda_maxwell), outputFolder=outputFolder,scale='log')

Constrained model: 

   model_w
   model_psi_a_0
      Psi_a_1_1_1:
         kernel constrained
         no bias constraint
      Psi_a_1_2_1:
         kernel constrained
         no bias constraint
      Psi_a_1_3_1:
         kernel constrained
         no bias constraint
      Psi_a_1_1_2:
         kernel constrained
      Psi_a_1_2_2:
         kernel constrained
      Psi_a_1_3_2:
         kernel constrained
   model_psi_a_1
      Psi_a_2_1_1:
         kernel constrained
         no bias constraint
      Psi_a_2_2_1:
         kernel constrained
         no bias constraint
      Psi_a_2_3_1:
         kernel constrained
         no bias constraint
      Psi_a_2_1_2:
         kernel constrained
      Psi_a_2_2_2:
         kernel constrained
      Psi_a_2_3_2:
         kernel constrained
   model_Psi
   model_tau_0
      Tau_1_1_1:
         no kernel constraint
         no bias constraint
      Tau_1_2_1:
         no kernel constraint
         no bias constraint
      Tau_1_3_1:
       

c:\Users\Kian\anaconda3\envs\TF\lib\site-packages\scipy\optimize\_minimize.py:571: RuntimeWarning:

Method L-BFGS-B does not use Hessian-vector product information (hessp).




Epoch 21: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-193626\ckpt\ckpt-epoch-21.ckpt

Epoch 41: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-193626\ckpt\ckpt-epoch-41.ckpt

Epoch 61: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-193626\ckpt\ckpt-epoch-61.ckpt

Epoch 81: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-193626\ckpt\ckpt-epoch-81.ckpt

Epoch 101: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-193626\ckpt\ckpt-epoch-101.ckpt

Epoch 121: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-193626\ckpt\ckpt-epoch-121.ckpt

Epoch 141: saving model to C:\Users\Kian\Documents\Projekte\ViscoCANN\vCANN\Results_anisotropic_multistep\20260109-1936

In [34]:
Outputs.plotLoss(history, Maxwell_monitor=Maxwell_monitor, title='$\Lambda = {:}$'.format(lambda_maxwell), outputFolder=outputFolder,scale='log')


In [ ]:
#%% Plot graphs and save model (summary)

Outputs.showModelSummary(model_fit, outputFolder)
Outputs.plotModelGraph(model_fit, outputFolder)
Outputs.saveModel(model_fit, outputFolder)

from exportArchitecture import exportArchitecture
exportArchitecture(model_fit, outputFolder, weights_local=False)
exportArchitecture(model_fit, outputFolder, weights_local=True)

#%%
import matplotlib.pyplot as plt
# loadedModel = Outputs.loadModel(outputFolder, 'model', custom_objects=custom_objects)





Model: "model_50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 F_input (InputLayer)           [(None, 1000, 3, 3)  0           []                               
                                ]                                                                 
                                                                                                  
 model_dir (dirModelOrtho)      (None, None, None,   0           ['F_input[0][0]']                
                                None)                                                             
                                                                                                  
 F_ref (Lambda)                 (None, None, None,   0           ['F_input[0][0]']                
                                None)                                                  

In [185]:
import matplotlib.pyplot as plt

def plot_strain_energy_factors_and_relaxation_times(Tau, Psi_a, Psi, lam, title, numTens):

    fig, axes = plt.subplots(numTens,2, figsize=(12,5*numTens))
    fig.suptitle(title)

    if numTens == 1:
        axes = np.array([axes])

    for ii in range(numTens):
        axes[ii,0].plot(lam[1:], Psi_a[ii][0,1:,0]/Psi[0,1:,ii], label='$\\beta_{1}$')
        axes[ii,0].plot(lam[1:], Psi_a[ii][0,1:,1]/Psi[0,1:,ii], label='$\\beta_{2}$')
        axes[ii,0].plot(lam[1:], Psi_a[ii][0,1:,2]/Psi[0,1:,ii], label='$\\beta_{3}$')
        axes[ii,0].plot(lam[1:], 0.14*np.ones_like(lam[1:]), color='tab:blue', linestyle='--', label='$\\beta_{1}^{ref}$')
        axes[ii,0].plot(lam[1:], 0.11*np.ones_like(lam[1:]), color='tab:orange', linestyle='--', label='$\\beta_{2}^{ref}$')
        axes[ii,0].plot(lam[1:], 0.11*np.ones_like(lam[1:]), color='tab:green', linestyle='--', label='$\\beta_{3}^{ref}$')
        axes[ii,0].grid()
        axes[ii,0].legend(fontsize=16)
        axes[ii,0].set_title('Strain energy factors')

        axes[ii,1].plot(lam,Tau[ii][0,:,0], color='tab:blue', label='$\\tau_1$')
        axes[ii,1].plot(lam,Tau[ii][0,:,1], color='tab:orange', label='$\\tau_2$')
        axes[ii,1].plot(lam,Tau[ii][0,:,2], color='tab:green', label='$\\tau_3$')
        axes[ii,1].plot(lam, np.ones_like(lam) * 1.00, color='tab:blue', linestyle='--', label='$\\tau_1^{ref}$')
        axes[ii,1].plot(lam, np.ones_like(lam) * 10.98, color='tab:orange', linestyle='--', label='$\\tau_2^{ref}$')
        axes[ii,1].plot(lam, np.ones_like(lam) * 50.01, color='tab:green', linestyle='--', label='$\\tau_3^{ref}$')
        axes[ii,1].set_yscale('log')
        axes[ii,1].grid()
        axes[ii,1].legend(fontsize=16)
        axes[ii,1].set_title('Relaxation times')

        plt.tight_layout()

### Equibiaxial tension
stretch_max = 1.5
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)

F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = lam
F[:,:,1,1] = lam
F[:,:,2,2] = 1.0/lam**2

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]
Tau = res[17]

plot_strain_energy_factors_and_relaxation_times(Tau, Psi_a, Psi, lam, title='Equibiaxial Tension', numTens=numTens)

### strip x
stretch_max = 2.5
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)

F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = lam
F[:,:,1,1] = 1.0
F[:,:,2,2] = 1.0/lam

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]
Tau = res[17]

plot_strain_energy_factors_and_relaxation_times(Tau, Psi_a, Psi, lam, title='Strip x', numTens=numTens)

### strip y
stretch_max = 2.5
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)


F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = 1.0
F[:,:,1,1] = lam
F[:,:,2,2] = 1.0/lam

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]
Tau = res[17]

plot_strain_energy_factors_and_relaxation_times(Tau, Psi_a, Psi, lam, title='Strip y', numTens=numTens)


### off x
stretch_max = 2.5
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)


F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = lam
F[:,:,1,1] = np.sqrt(lam)
F[:,:,2,2] = 1.0/(lam*np.sqrt(lam))

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]
Tau = res[17]

plot_strain_energy_factors_and_relaxation_times(Tau, Psi_a, Psi, lam, title='Off x', numTens=numTens)


### off y
stretch_max = 2.5
stretch_min = 1
lam_dot = 0.1
lam = np.linspace(stretch_min,stretch_max,nSteps)


F = np.zeros([1, len(lam), 3, 3])
F[:,:,0,0] = np.sqrt(lam)
F[:,:,1,1] = lam
F[:,:,2,2] = 1.0/(lam*np.sqrt(lam))

total_time = (stretch_max-stretch_min)/lam_dot*2
time = np.linspace(0,total_time,len(lam)+1)[:-1].reshape(1,-1)

inp=[F,time]
res = model_full.predict(inp)

invars = res[0]
Psi_a = res[-11]
Psi = res[1]
Tau = res[17]

plot_strain_energy_factors_and_relaxation_times(Tau, Psi_a, Psi, lam, title='Off y', numTens=numTens)

1/1 [==============================] - 0s 108ms/step


In [80]:
S_i = res[19]
S_i[1]

S_a_neq = res[20]
S_a_neq[1]

array([[[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        ...,

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]]])

In [298]:
n = 1
F[0,n]


S_infty = res[16]
Tau = res[17]
S = res[18][0,n] 
# S_neq = res[39]
dPsi_a_dI_ref = res[25][1]
dPsi_a_dJ_ref = res[26][2]
alpha_a = res[28]
beta_a = res[29]
S_a = res[32]
Q_unstack = res[33]
P = res[34]
# dPsi_a_dI_c = res[-9]
# dPsi_a_dJ_c = res[-8]
# ddPsi_a_ddI_c = res[-7]
# ddPsi_a_ddJ_c = res[-6]
# ddPsi_a_dIdJ_c = res[-5]
# S_ra_neq_bar = res[38]
# dSdCbar = res[35]
# dS_a_dC_bar = res[36]
# dS_a_neq_bar_1, dS_a_neq_bar_2, dS_a_neq_bar_3 = res[-3], res[-2], res[-1]

# Q_unstack[2][0,]
# time[0,n]
#S_a[0,n]
#Tau[0,n]
# dPsi_a_dI_c[0,n]
# print(S_ra_neq_bar[0,n,0])
# print(dS_a_neq_bar_1[n][0,n])
# print(dS_a_neq_bar_2[n][0,n])
# print(dS_a_neq_bar_3[n][0,n])
# Tau[0,n]
# print((dS_a_dC_bar[0][0,n,0,0,:,:] + dS_a_dC_bar[0][0,n,0,0,:,:] + dS_a_dC_bar[0][0,n,0,0,:,:])*2)

# # dQdCbar_2[0][0,0]
# print(dS_a_neq_bar_3[n][0,n,2,2])

In [35]:
# ckpt = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_Hossain\\20251030-185939\\ckpt\\ckpt-epoch-1001.ckpt"
# ckpt = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_Hossain\\20251103-190018\\ckpt\\ckpt-epoch-561.ckpt"
# loaded_model = Outputs.loadModel(outputFolder, 'model', custom_objects)
# model_fit.load_weights(ckpt)
#%% individial plotting
    
#%% Structural Tensor
# Plots.plot_struc_tensor(model_fit, plotly=False, outputFolder=outputFolder)

#%% Stress Response 
# Plots.plot_multi_step_isotropic(model_fit, trainDs, outputFolder)
# Plots.plot_stress(model_fit, trainDs, valDs, outputFolder)
Plots.plot_multi_step(model_fit, trainDs, outputFolder)

        

1/1 [==============================] - 0s 145ms/step


In [ ]:
from checkpoint_analysis import parallel_checkpoint_analysis
parallel_checkpoint_analysis(model_fit, pathToData, outputFolder, rateDependent, checkpoint_dir=None, max_workers=None)

In [ ]:
from checkpoint_analysis import create_video_from_checkpoint_analysis
video_folder = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_General_biaxial\\20251125-015147"
create_video_from_checkpoint_analysis(video_folder, fps=2, duration_per_frame=0.5)